# 🧠📦 ThinkDome: Kaggle Notebook Deployment Guide

This notebook sets up and runs the **ThinkDome Autonomous LLM Sandbox & Tool Orchestrator** inside a Kaggle notebook environment.

### ⚠️ Important Note on Kaggle Environment
Kaggle kernels run inside containers and **do not support Docker-in-Docker**. This setup uses the `subprocess` backend.

## 🛠️ Step 1: Install ThinkDome Package

In [ ]:
!pip install -q git+https://github.com/abeldirectory252/ThinkDome.git
print('✅ ThinkDome installed')

## 🐍 Step 2: Test the Programmatic SDK API

In [ ]:
from thinkdome import Sandbox

with Sandbox(backend="subprocess", timeout=10) as dome:
    result = dome.run("print('Hello from inside ThinkDome SDK!')")
    print("Success:", result.success)
    print("Output:", result.output.strip())
    print("Duration (ms):", result.duration_ms)

## ⚙️ Step 3: Configure Environment Variables

In [ ]:
import os
os.environ['EXECUTOR_BACKEND'] = 'subprocess'
os.environ['API_KEY'] = 'sk_tb_admin_super_secret_override_token'
os.environ['FILE_STORAGE_DIR'] = './storage'
os.makedirs('./storage', exist_ok=True)
print('Environment variables configured.')

## 🚀 Step 4: Run ThinkDome API Server in Background

In [ ]:
import subprocess, time, sys

print('Starting ThinkDome API server...')
server_process = subprocess.Popen(
    [sys.executable, '-m', 'thinkdome.cli', 'serve', '--host', '127.0.0.1', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
time.sleep(5)
if server_process.poll() is None:
    print('✅ ThinkDome is running!')
else:
    print('❌ Server failed:', server_process.communicate()[1])

## 🔍 Step 5: Test API Endpoints

In [ ]:
import httpx
BASE_URL = 'http://127.0.0.1:8000'
resp = httpx.get(f'{BASE_URL}/health')
print('Health:', resp.status_code, resp.json())
schema_resp = httpx.get(f'{BASE_URL}/orchestrator_schema.json')
print('Tools loaded:', len(schema_resp.json()))

## 🔐 Step 5.5: Register User & Login

In [ ]:
reg = httpx.post(f'{BASE_URL}/v1/auth/register',
                 json={'username': 'kaggle_operator', 'password': 'secure_password_123'})
print('Register:', reg.status_code, reg.json())

login = httpx.post(f'{BASE_URL}/v1/auth/login',
                   json={'username': 'kaggle_operator', 'password': 'secure_password_123'})
jwt_token = login.json().get('access_token')
print('🔑 JWT Token:', jwt_token[:40] + '...' if jwt_token else 'None')

## 🔑 Step 6: Create API Keys

In [ ]:
SUPER_KEY = os.environ.get('API_KEY')
headers = {'Authorization': f'Bearer {SUPER_KEY}'}

llm_key_resp = httpx.post(f'{BASE_URL}/v1/admin/keys', headers=headers,
                          json={'display_name': 'Kaggle LLM Agent', 'token_type': 'LLM'})
llm_token = llm_key_resp.json()['token']
print('🔑 LLM Token:', llm_token)

admin_key_resp = httpx.post(f'{BASE_URL}/v1/admin/keys', headers=headers,
                            json={'display_name': 'Kaggle Admin', 'token_type': 'ADMIN'})
admin_token = admin_key_resp.json()['token']
print('🔑 Admin Token:', admin_token)

## ⚙️ Step 7: Tool Orchestrator Tests

In [ ]:
def call_tool(name, inputs, token):
    payload = {'type': 'tool_use', 'id': f'call_{name}', 'name': name, 'input': inputs}
    return httpx.post(f'{BASE_URL}/v1/orchestrate',
                      headers={'Authorization': f'Bearer {token}'}, json=payload).json()

# LLM token: memory_store (allowed)
r1 = call_tool('memory_store', {'key': 'note', 'content': 'Running on Kaggle', 'tags': ['kaggle']}, llm_token)
print('memory_store:', r1)

# LLM token: shell_exec (denied)
r2 = call_tool('shell_exec', {'command': 'whoami'}, llm_token)
print('shell_exec (LLM - denied):', r2.get('content', '')[:80])

# Admin token: shell_exec (allowed)
r3 = call_tool('shell_exec', {'command': 'ls -la'}, admin_token)
print('shell_exec (Admin):', r3.get('content', '')[:120])

## 🌐 Step 8: Expose Externally (Optional)

In [ ]:
!npm install -g localtunnel
print('Run: lt --port 8000')

## 🛑 Step 9: Shutdown Server

In [ ]:
server_process.terminate()
print('ThinkDome server stopped.')

---
# 🤖 Part 2: Gemma 2 Agent with Sandbox Tools

Load **Google Gemma-2-2B-Instruct** in 4-bit quantization and use ThinkDome sandbox tools (`run_code`, `web_search`, `list_dir`, `read_file`) as an agentic LLM.

**Requirements:** Enable **GPU (T4)** in Kaggle. Accept Gemma license at https://huggingface.co/google/gemma-2-2b-it

In [ ]:
# Install 4-bit quantization dependencies
!pip install -q bitsandbytes accelerate transformers torch
print('✅ Quantization libs installed')

In [ ]:
# Create a high-resource sandbox for LLM agent work
import httpx, os, json, subprocess, time, sys
BASE_URL = 'http://127.0.0.1:8000'
ADM = {'Authorization': f"Bearer {os.environ.get('API_KEY', 'sk_tb_admin_super_secret_override_token')}"}

# Auto-start server if not running
server_running = False
try:
    resp = httpx.get(f'{BASE_URL}/health', timeout=2.0)
    if resp.status_code == 200:
        server_running = True
        print('✅ ThinkDome server is already running.')
except Exception:
    pass

if not server_running:
    print('🚀 ThinkDome server is not running. Starting it in the background...')
    os.environ['EXECUTOR_BACKEND'] = 'subprocess'
    os.environ['API_KEY'] = 'sk_tb_admin_super_secret_override_token'
    os.environ['FILE_STORAGE_DIR'] = './storage'
    os.makedirs('./storage', exist_ok=True)
    
    server_process = subprocess.Popen(
        [sys.executable, '-m', 'thinkdome.cli', 'serve', '--host', '127.0.0.1', '--port', '8000'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    time.sleep(5)
    if server_process.poll() is None:
        print('✅ ThinkDome server successfully started!')
    else:
        print('❌ Failed to start server:', server_process.communicate()[1])

# Create admin token for agent
ak = httpx.post(f'{BASE_URL}/v1/admin/keys', headers=ADM,
                json={'display_name': 'Gemma Agent', 'token_type': 'ADMIN'})
AGENT_TOKEN = ak.json()['token']

# Create sandbox
sb = httpx.post(f'{BASE_URL}/v1/admin/sandboxes', headers=ADM,
                json={'name': 'Gemma Agent Sandbox', 'memory_mb': 4096,
                      'cpu_cores': 4.0, 'timeout_sec': 120, 'network_enabled': True}).json()
SANDBOX_ID = sb['sandbox_id']
print(f'📦 Sandbox: {sb["name"]} | {sb["memory_mb"]}MB | {sb["cpu_cores"]}vCPU | ID={SANDBOX_ID}')

In [ ]:
# Load Gemma 2 2B Instruct in 4-bit
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'google/gemma-2-2b-it'
# Set HF_TOKEN in Kaggle secrets or uncomment:
# os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN'
HF_TOKEN = os.environ.get('HF_TOKEN', None)

print(f'Loading {MODEL_ID} in 4-bit...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto', token=HF_TOKEN)

print(f'✅ Loaded! Memory: {model.get_memory_footprint()/1e9:.2f}GB | Device: {model.device}')
if torch.cuda.is_available(): print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Define the agentic tool-calling loop
import re

TOOLS_DESC = """You have access to sandbox tools. To use one, output EXACTLY:
<tool_call>
{"name": "TOOL_NAME", "input": {PARAMS}}
</tool_call>

Tools:
1. run_code - Execute Python. Params: {"code": "..."}
2. web_search - Search web. Params: {"query": "...", "max_results": 5}
3. list_dir - List files. Params: {"path": "."}
4. read_file - Read file. Params: {"path": "file.py"}

Use run_code for math/data/code tasks. After tool results, give your final answer.
"""

def sandbox_tool(name, inputs):
    resp = httpx.post(f'{BASE_URL}/v1/orchestrate',
        headers={'Authorization': f'Bearer {AGENT_TOKEN}', 'X-Sandbox-Id': SANDBOX_ID},
        json={'type':'tool_use','id':f'agent_{name}','name':name,'input':inputs}, timeout=130.0)
    content = resp.json().get('content', '')
    try:
        p = json.loads(content)
        if isinstance(p, dict) and 'stdout' in p:
            return p['stdout'] + (p.get('stderr','') if p.get('exit_code',0)!=0 else '')
        return json.dumps(p, indent=2)[:2000]
    except: return str(content)[:2000]

def gen(messages, max_tokens=512):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.7, do_sample=True, top_p=0.9)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def agent_ask(question, max_rounds=3):
    msgs = [{'role':'user','content': TOOLS_DESC+'\nQuestion: '+question}]
    for i in range(max_rounds):
        print(f'\n--- Round {i+1} ---')
        resp = gen(msgs)
        print(f'🤖 LLM: {resp[:500]}')
        match = re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', resp, re.DOTALL)
        if not match: return resp
        try:
            tc = json.loads(match.group(1))
            print(f'🔧 Tool: {tc["name"]}({json.dumps(tc["input"])[:100]})')
            result = sandbox_tool(tc['name'], tc['input'])
            print(f'📋 Result: {result[:300]}')
            msgs.append({'role':'model','content':resp})
            msgs.append({'role':'user','content':f'Tool result ({tc["name"]}):\n{result}\n\nProvide your final answer.'})
        except Exception as e:
            print(f'⚠️ Parse error: {e}'); return resp
    return gen(msgs)

print('✅ Agent ready!')

In [ ]:
# Test 1: Math via sandbox code execution
answer = agent_ask('What is the 50th Fibonacci number? Use Python code to calculate it.')
print('\n' + '='*60 + '\nFINAL:', answer)

In [ ]:
# Test 2: Web search
answer = agent_ask('Search the web for the latest Python version and tell me about it.')
print('\n' + '='*60 + '\nFINAL:', answer)

In [ ]:
# Test 3: File exploration
answer = agent_ask('List files in the current directory and read README.md. What is this project?')
print('\n' + '='*60 + '\nFINAL:', answer)

In [ ]:
# Test 4: Complex code task
answer = agent_ask('Write a Python function to check if a number is prime, then find all primes between 100 and 150.')
print('\n' + '='*60 + '\nFINAL:', answer)

In [ ]:
# Cleanup: terminate sandbox
httpx.delete(f'{BASE_URL}/v1/admin/sandboxes/{SANDBOX_ID}', headers=ADM)
print('✅ Gemma agent sandbox terminated.')